
# Random Fourier Features (RFF): Theory (LaTeX) + Code Demos

**Goals**
- ✅ Verify LaTeX renders in Markdown.
- ✅ Step‑by‑step math for RFF: spectral measure, cosine identities, feature maps.
- ✅ Construct **kernel matrices**: exact Gaussian (RBF) vs **RFF-approximated**; print and compare.
- ✅ Show approximation improves as feature count \(D\) increases.
- ✅ Use RFF for least-squares / ridge and compare with exact Kernel Ridge (KRR).



**LaTeX render test:** \( e^{i\pi} + 1 = 0 \)  &nbsp;&nbsp; and &nbsp;&nbsp; 
\[
\int_{-\infty}^{\infty} e^{-x^2}\,dx \;=\; \sqrt{\pi}.
\]
If you see nicely rendered math, you're good to go.



## 1) Shift‑Invariant Kernels, Bochner’s Theorem, and Spectral Measure

A kernel \(k: \mathbb{R}^d \times \mathbb{R}^d \to \mathbb{R}\) is **shift‑invariant** if
\[
k(x, x') \;=\; k(x - x') \;\; \text{for all } x,x' \in \mathbb{R}^d.
\]
Let \(\delta = x - x'\). For the **Gaussian/RBF kernel** with lengthscale \(\ell>0\),
\[
k_{\text{RBF}}(x,x') \;=\; \exp\!\Big(-\frac{\lVert x - x'\rVert^2}{2\ell^2}\Big) \;=\; k_{\text{RBF}}(\delta).
\]

**Bochner’s theorem (informal)**: A continuous, shift‑invariant, positive‑definite kernel \(k(\delta)\) is the **Fourier transform** of a non‑negative measure \(p(\omega)\) on \(\mathbb{R}^d\):
\[
k(\delta) \;=\; \int_{\mathbb{R}^d} e^{i\,\omega^\top \delta}\, p(\omega)\, d\omega.
\]
For the RBF kernel,
\[
p(\omega) \;=\; \mathcal{N}\!\left(0,\; \tfrac{1}{\ell^2} I_d\right).
\]
Because \(k\) is real and even, we can write
\[
k(\delta) \;=\; \int_{\mathbb{R}^d} \cos(\omega^\top \delta)\, p(\omega)\, d\omega.
\]



## 2) From Spectral Integral to Features (Cosine Identities)

A Monte‑Carlo approximation of the integral uses random draws \(\omega_j \sim p(\omega)\).
A convenient **cosine feature** representation uses a random phase \(b_j \sim \mathrm{Unif}[0,2\pi]\):
\[
\phi(x) \;=\; \sqrt{\frac{2}{D}}\;\big[\cos(\omega_1^\top x + b_1),\;\ldots,\;\cos(\omega_D^\top x + b_D)\big]^\top.
\]
Then the inner product
\[
\phi(x)^\top \phi(x') \;=\; \frac{2}{D}\sum_{j=1}^D \cos(\omega_j^\top x + b_j)\,\cos(\omega_j^\top x' + b_j).
\]

**Cosine product identity**:
\[
\cos u\,\cos v \;=\; \tfrac{1}{2}\big(\cos(u-v) + \cos(u+v)\big).
\]
Let \(u=\omega_j^\top x + b_j\) and \(v=\omega_j^\top x' + b_j\). Then
\[
\cos(\omega_j^\top x + b_j)\cos(\omega_j^\top x' + b_j)
= \tfrac{1}{2}\Big(\cos\!\big(\omega_j^\top(x - x')\big) + \cos\!\big(\omega_j^\top(x + x') + 2b_j\big)\Big).
\]
Taking expectation over \(b_j \sim \mathrm{Unif}[0,2\pi]\) kills the second term:
\[
\mathbb{E}_{b_j}\big[\cos(\omega_j^\top x + b_j)\cos(\omega_j^\top x' + b_j)\big]
= \tfrac{1}{2}\cos\!\big(\omega_j^\top(x - x')\big).
\]
Therefore
\[
\mathbb{E}_{\omega,b}\big[\phi(x)^\top \phi(x')\big]
= \frac{2}{D}\sum_{j=1}^D \tfrac{1}{2}\,\mathbb{E}_{\omega_j}\!\big[\cos(\omega_j^\top (x-x'))\big]
= \mathbb{E}_{\omega}\!\big[\cos(\omega^\top (x-x'))\big]
= k(x,x').
\]
So \( \phi(x)^\top \phi(x') \) is an **unbiased Monte‑Carlo estimator** of \(k(x,x')\), and its variance shrinks with \(D\).



## 3) Kernel Matrices (Exact vs RFF)

Given data \(X = [x_1,\dots,x_n]^\top \in \mathbb{R}^{n\times d}\):

- The **exact** kernel matrix \(K \in \mathbb{R}^{n\times n}\) has entries
\[
K_{ij} \;=\; k(x_i, x_j).
\]

- Draw \(\omega_1,\dots,\omega_D \sim p(\omega)\) and \(b_1,\dots,b_D \sim \mathrm{Unif}[0,2\pi]\). Define the **feature matrix** \(Z\in\mathbb{R}^{n\times D}\) by
\[
Z_{i j} \;=\; \sqrt{\frac{2}{D}}\;\cos(\omega_j^\top x_i + b_j).
\]
Then the **RFF kernel approximation** is
\[
\widehat{K} \;=\; Z Z^\top,\qquad \widehat{K}_{ij} \;=\; \phi(x_i)^\top \phi(x_j).
\]
We often monitor the **relative Frobenius error**
\[
\frac{\lVert K - \widehat{K}\rVert_F}{\lVert K\rVert_F}.
\]


## 4) Setup (imports) & helpers

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import time

rng = np.random.default_rng(42)

def rbf_kernel(X, Y, lengthscale=1.0):
    """Pairwise RBF kernel between X (nxd) and Y (mxd). Returns (n x m)."""
    X = np.atleast_2d(X)
    Y = np.atleast_2d(Y)
    X_norm = np.sum(X**2, axis=1)[:, None]
    Y_norm = np.sum(Y**2, axis=1)[None, :]
    sqdist = X_norm + Y_norm - 2 * X @ Y.T
    return np.exp(-0.5 * sqdist / (lengthscale**2))

def sample_rff_params(d, D, lengthscale=1.0, seed=None):
    """Sample (omega, b) for RFF features for RBF kernel."""
    rng_local = np.random.default_rng(seed)
    omega = rng_local.normal(loc=0.0, scale=1.0/lengthscale, size=(D, d))  # D x d
    b = rng_local.uniform(low=0.0, high=2*np.pi, size=(D,))
    return omega, b

def rff_features(X, omega, b):
    """Compute RFF features phi(X) with given (omega, b)."""
    proj = X @ omega.T  # n x D
    Z = np.sqrt(2.0/omega.shape[0]) * np.cos(proj + b)
    return Z

def ridge_closed_form(Z, y, lam=1e-3):
    """Closed-form ridge: beta = (Z^T Z + lam I)^(-1) Z^T y"""
    n, D = Z.shape
    A = Z.T @ Z + lam * np.eye(D)
    b = Z.T @ y
    beta = np.linalg.solve(A, b)
    return beta

def krr_predict(X_train, y_train, X_test, lengthscale=1.0, lam=1e-3):
    """Kernel Ridge Regression prediction with RBF kernel using closed form."""
    K = rbf_kernel(X_train, X_train, lengthscale=lengthscale)
    n = K.shape[0]
    alpha = np.linalg.solve(K + lam * np.eye(n), y_train)
    K_s = rbf_kernel(X_test, X_train, lengthscale=lengthscale)
    return K_s @ alpha



## 5) Printed Comparison: Exact RBF \(K\) vs RFF \(\widehat{K}=ZZ^\top\)

We keep \(n\) small so matrices are readable. Adjust \(n_{\text{print}}\), \(D\), and \(\ell\) to see how the approximation behaves.


In [ ]:

# === Printed comparison settings ===
n_print = 8
d = 3
D = 128
lengthscale = 1.25
seed = 123

np.set_printoptions(precision=3, suppress=True)

X_small = rng.normal(size=(n_print, d))

# Exact kernel
K = rbf_kernel(X_small, X_small, lengthscale=lengthscale)

# RFF approx
omega, b = sample_rff_params(d, D, lengthscale=lengthscale, seed=seed)
Z = rff_features(X_small, omega, b)
K_hat = Z @ Z.T

# Differences & metrics
diff = K - K_hat
fro_rel = np.linalg.norm(diff, 'fro') / np.linalg.norm(K, 'fro')
max_abs = np.max(np.abs(diff))

print("Exact Gaussian (RBF) kernel K:")
print(K, "\n")
print("RFF kernel approximation K_hat = Z Z^T (D={}):".format(D))
print(K_hat, "\n")
print("Difference (K - K_hat):")
print(diff, "\n")
print(f"Relative Frobenius error: {fro_rel:.6f}")
print(f"Max absolute entry-wise error: {max_abs:.6f}")



## 6) Demo A — Approximation quality vs feature count \(D\)

We track the **relative Frobenius error** \( \lVert K - \widehat{K}\rVert_F / \lVert K\rVert_F \) as \(D\) increases.


In [ ]:

n = 300            # number of points
d = 5              # input dimension
lengthscale = 1.25
D_list = [16, 32, 64, 128, 256, 512, 1024]

X = rng.normal(size=(n, d))
K = rbf_kernel(X, X, lengthscale=lengthscale)
K_norm = np.linalg.norm(K, ord='fro')

errors = []
for D in D_list:
    omega, b = sample_rff_params(d, D, lengthscale=lengthscale, seed=123)
    Z = rff_features(X, omega, b)
    K_hat = Z @ Z.T
    err = np.linalg.norm(K - K_hat, ord='fro') / K_norm
    errors.append(err)

plt.figure()
plt.plot(D_list, errors, marker='o')
plt.xscale('log', base=2)
plt.xlabel('Number of features D (log scale)')
plt.ylabel('Relative Frobenius error ||K - K_hat||_F / ||K||_F')
plt.title('RFF kernel approximation vs D')
plt.show()



## 7) Demo B — Least Squares / Ridge with RFF vs exact Kernel Ridge (KRR)

We construct a nonlinear regression and compare **KRR (exact RBF)** to **RFF + ridge** for several \(D\). We report test MSE and fit time.


In [ ]:

# Synthetic non-linear regression task
n_train, n_test, d = 400, 300, 5
X_train = rng.uniform(-3, 3, size=(n_train, d))
X_test  = rng.uniform(-3, 3, size=(n_test, d))

def true_fun(X):
    r = np.linalg.norm(X, axis=1)
    return np.sin(r) + 0.2*np.cos(2*r)

y_train = true_fun(X_train) + 0.1 * rng.normal(size=n_train)
y_test  = true_fun(X_test)  + 0.1 * rng.normal(size=n_test)

lengthscale = 1.2
lam = 1e-2
D_list = [32, 64, 128, 256, 512, 1024]

# Exact KRR (reference)
t0 = time.time()
K = rbf_kernel(X_train, X_train, lengthscale=lengthscale)
alpha = np.linalg.solve(K + lam * np.eye(n_train), y_train)
K_s = rbf_kernel(X_test, X_train, lengthscale=lengthscale)
y_pred_krr = K_s @ alpha
krr_time = time.time() - t0
krr_mse = np.mean((y_pred_krr - y_test)**2)

rff_mse = []
rff_time = []
for D in D_list:
    omega, b = sample_rff_params(d, D, lengthscale=lengthscale, seed=999)
    Z_train = rff_features(X_train, omega, b)
    Z_test = rff_features(X_test, omega, b)

    t0 = time.time()
    beta = ridge_closed_form(Z_train, y_train, lam=lam)
    rff_time.append(time.time() - t0)
    y_pred = Z_test @ beta
    rff_mse.append(np.mean((y_pred - y_test)**2))

print(f"KRR:    MSE={krr_mse:.4f}, fit_time={krr_time:.4f} s")
for D, mse, tt in zip(D_list, rff_mse, rff_time):
    print(f"RFF D={D:4d}: MSE={mse:.4f}, fit_time={tt:.6f} s")

plt.figure()
plt.plot(D_list, rff_mse, marker='o', label='RFF + Ridge (test MSE)')
plt.axhline(krr_mse, linestyle='--', label='KRR reference (test MSE)')
plt.xscale('log', base=2)
plt.xlabel('Number of features D (log scale)')
plt.ylabel('Test MSE')
plt.title('RFF Ridge vs exact KRR')
plt.legend()
plt.show()



## 8) What to try next

- **Adjust printing demo:** change \(n_{\text{print}}\), \(D\), \(\ell\) to see entries converge.
- Try the **cos+sin** two‑channel map instead of random phase \(b\).
- Add **classification** demo (logistic regression) on \(Z\).
- Compare with **Orthogonal Random Features (ORF)** to reduce variance.
- Cross‑validate \(\ell\) and \(\lambda\) for best generalization.
